# Data Preprocessing Basics

Before feeding data to ML models, we need to clean and prepare it. This notebook covers the essential preprocessing steps.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

## Sample Dataset

Let's create a sample dataset with common real-world issues.

In [ ]:
# Create sample data with issues
data = {
    'Age': [25, 30, np.nan, 35, 40, 28, np.nan, 45, 32, 38],
    'Salary': [50000, 60000, 55000, np.nan, 80000, 52000, 58000, 90000, np.nan, 75000],
    'City': ['Delhi', 'Mumbai', 'Delhi', 'Bangalore', 'Mumbai', 'Delhi', 'Bangalore', 'Mumbai', 'Delhi', 'Bangalore'],
    'Gender': ['M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F'],
    'Purchased': [0, 1, 0, 1, 1, 0, 0, 1, 0, 1]
}

df = pd.DataFrame(data)
df

## 1. Handling Missing Values

Missing values can break ML algorithms. Common strategies:
- **Drop** rows/columns with missing values
- **Fill** with mean, median, mode, or a constant

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Option 1: Drop rows with missing values
df_dropped = df.dropna()
print(f"Original rows: {len(df)}")
print(f"After dropping: {len(df_dropped)}")

In [ ]:
# Option 2: Fill with mean (for numerical columns)
df_filled = df.copy()
df_filled['Age'] = df_filled['Age'].fillna(df_filled['Age'].mean())
df_filled['Salary'] = df_filled['Salary'].fillna(df_filled['Salary'].median())

print("After filling missing values:")
df_filled

In [ ]:
# Using sklearn SimpleImputer
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')  # can be 'median', 'most_frequent', 'constant'

# Only for numerical columns
numerical_cols = ['Age', 'Salary']
df[numerical_cols] = imputer.fit_transform(df[numerical_cols])
df

## 2. Encoding Categorical Variables

ML models need numbers. We convert categories to numbers.

### Label Encoding
Assigns a number to each category. Use for **ordinal** data (has order) or **binary** categories.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Good for binary categories like Gender
le = LabelEncoder()
df['Gender_encoded'] = le.fit_transform(df['Gender'])

print("Label Encoding for Gender:")
print(df[['Gender', 'Gender_encoded']])
print(f"\nClasses: {le.classes_}")

### One-Hot Encoding
Creates binary columns for each category. Use for **nominal** data (no order).

In [ ]:
# One-Hot Encoding for City (no inherent order)
df_encoded = pd.get_dummies(df, columns=['City'], prefix='City')
df_encoded

In [ ]:
# Using sklearn OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' avoids dummy variable trap
city_encoded = ohe.fit_transform(df[['City']])

print("One-Hot Encoded City (dropped first):")
print(city_encoded)
print(f"Feature names: {ohe.get_feature_names_out()}")

## 3. Feature Scaling

Features with different scales can cause problems. Some algorithms (like KNN, SVM, Neural Networks) are sensitive to scale.

### Standardization (Z-score)
Transforms data to have mean=0 and std=1.

Formula: `z = (x - mean) / std`

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Scale Age and Salary
df_scaled = df.copy()
df_scaled[['Age_scaled', 'Salary_scaled']] = scaler.fit_transform(df[['Age', 'Salary']])

print("Before scaling:")
print(df[['Age', 'Salary']].describe().loc[['mean', 'std']])

print("\nAfter StandardScaler:")
print(df_scaled[['Age_scaled', 'Salary_scaled']].describe().loc[['mean', 'std']])

### Min-Max Normalization
Scales data to a fixed range (usually 0-1).

Formula: `x_norm = (x - min) / (max - min)`

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()

df_scaled[['Age_minmax', 'Salary_minmax']] = minmax.fit_transform(df[['Age', 'Salary']])

print("After MinMaxScaler:")
print(df_scaled[['Age_minmax', 'Salary_minmax']].describe().loc[['min', 'max']])

### When to use which?

| Scaler | When to use |
|--------|-------------|
| StandardScaler | When data is normally distributed, for algorithms that assume normal distribution |
| MinMaxScaler | When you need bounded values (0-1), for neural networks |
| No scaling | Tree-based models (Decision Tree, Random Forest, XGBoost) don't need scaling |

## 4. Train-Test Split (Important!)

**Always split BEFORE preprocessing** to avoid data leakage.

Data leakage: When information from test set influences training. This gives overly optimistic results.

In [ ]:
# Correct way: split first, then preprocess
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target

# Step 1: Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Fit scaler on training data ONLY
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)        # only transform (use training params)

print(f"Training set mean: {X_train_scaled.mean():.4f}")
print(f"Test set mean: {X_test_scaled.mean():.4f}")  # will be close to 0 but not exactly

## 5. Complete Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Create fresh sample data
data = {
    'Age': [25, 30, np.nan, 35, 40, 28, np.nan, 45, 32, 38],
    'Salary': [50000, 60000, 55000, np.nan, 80000, 52000, 58000, 90000, np.nan, 75000],
    'City': ['Delhi', 'Mumbai', 'Delhi', 'Bangalore', 'Mumbai', 'Delhi', 'Bangalore', 'Mumbai', 'Delhi', 'Bangalore'],
    'Purchased': [0, 1, 0, 1, 1, 0, 0, 1, 0, 1]
}
df = pd.DataFrame(data)

X = df.drop('Purchased', axis=1)
y = df['Purchased']

# Define preprocessing for different column types
numeric_features = ['Age', 'Salary']
categorical_features = ['City']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

# Combine into single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create full pipeline with model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Train
model.fit(X, y)

# Predict
print(f"Predictions: {model.predict(X)}")
print(f"Accuracy: {model.score(X, y):.2f}")

## Summary

| Step | What | When |
|------|------|------|
| Handle missing values | Impute or drop | Always check first |
| Encode categories | Label or One-Hot | For categorical columns |
| Scale features | Standardize or Normalize | For distance-based algorithms |
| Split data | Train/Test | Before any preprocessing |

**Remember**: Fit preprocessors on training data only, then transform both train and test.